In [2]:
"""
Low-Thrust Transfer: Earth → 2024 PDC25
Sims-Flanagan, pykep 2.x + pygmo NLopt + grss

Propulsion: 2× NASA NEXT-C ion engines (as used on Psyche mission)
            Replaces chemical HiPAT — rendezvous constraints match
            asteroid velocity at arrival so no insertion burn needed.

Mass budget (revised — no NTO/MMH/HiPAT):
  Old dry mass          : 1821.1 kg
  Remove chem propulsion:  -220.0 kg  (NTO/MMH tanks)
  Remove HiPAT hardware :    -6.0 kg
  Add ion engine hardware:  +50.0 kg  (2× NEXT-C + PPU + Xe tank)
  Revised dry mass      : 1645.0 kg
  Xenon propellant      :  500.0 kg
  Wet mass              : 2145.0 kg

Blast date: FIXED
grss: called ONCE at blast date
pygmo: NLopt SLSQP solver only
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from astropy.time import Time as AstroTime
import pykep as pk
import pygmo as pg
from grss import prop, utils

# ── CONFIG ────────────────────────────────────────────────────────
SOL_FILE   = './data/sol.json'
OUT_PLOT   = './data/low_thrust_result.png'

BLAST_DATE   = '2033-11-01'
T0_LOWER_ISO = '2030-03-01'
T0_UPPER_ISO = '2031-10-01'

# ── Spacecraft: 2× NASA NEXT-C ion engines ────────────────────────
# Replaces chemical system — rendezvous matches asteroid velocity
# so no HiPAT orbit insertion burn needed at arrival.
M_DRY  = 1901.1   # kg  revised dry (no NTO/MMH tanks, +ion hardware)
M_FUEL =  500.0   # kg  xenon propellant
M_WET  = M_DRY + M_FUEL   # 2145.0 kg

THRUST = 0.472    # N   2 × 236 mN NEXT-C (Psyche heritage)
ISP    = 4190     # s   NEXT-C specific impulse
NSEG   = 30       # Sims-Flanagan segments

# ── Falcon Heavy departure C3 ─────────────────────────────────────
# Falcon Heavy expendable delivers 2145 kg at C3 ~ 15 km²/s²
# (verified against NASA Launch Services Program performance curves)
# C3 = |v_infinity|²  →  v_inf = sqrt(C3)
# The spacecraft departs Earth with this excess velocity added
# in the prograde direction (best case — optimiser direction below).
# For PDC25 the departure is from ~1 AU so prograde is approximate.
C3_KM2S2 = 15.0   # km²/s²  — Falcon Heavy expendable at 2145 kg
VINF_MS   = np.sqrt(C3_KM2S2) * 1e3   # m/s  ≈ 3873 m/s

N_STARTS    = 25
RANDOM_SEED = 42

NONGRAV = {'a1':0.0,'a2':1.004247567426106e-13,'a3':0.0,
           'alpha':1.0,'k':0.0,'m':2.0,'n':0.0,'r0_au':1.0,'radius':75.0}
BODY_ID = '2024PDC25'
AU_M    = 1.495978707e11
DAY_S   = 86400.0

# ── GRSS: one call ────────────────────────────────────────────────
def asteroid_state_at(sol, t_mjd):
    ng = prop.NongravParameters()
    ng.a1=sol.get('a1',NONGRAV['a1']); ng.a1Est=False
    ng.a2=sol.get('a2',NONGRAV['a2']); ng.a2Est=True
    ng.a3=sol.get('a3',NONGRAV['a3']); ng.a3Est=False
    ng.alpha=NONGRAV['alpha']; ng.k=NONGRAV['k']
    ng.m=NONGRAV['m'];         ng.n=NONGRAV['n']
    ng.r0_au=NONGRAV['r0_au']
    sim = prop.PropSimulation(BODY_ID, sol['t'], 440, utils.default_kernel_path)
    sim.set_integration_parameters(t_mjd, tEval=[t_mjd], tEvalUTC=False,
                                    evalApparentState=False,
                                    convergedLightTime=False)
    cometary = [sol[k] for k in ['e','q','tp','om','w','i']]
    sim.add_integ_body(prop.IntegBody(BODY_ID, sol['t'], 0.0,
                                      NONGRAV['radius']/149597.871,
                                      cometary, ng))
    sim.integrate()
    s = np.array(sim.xIntegEval[0][:6])
    return s[:3]*AU_M, s[3:]*(AU_M/DAY_S)

# ── PYGMO UDP ─────────────────────────────────────────────────────
class LowThrustUDP:
    """
    Decision vector: [t0_mjd2000, u0x,u0y,u0z,...,mf_kg]
    TOF = blast_mjd2k - t0  (always arrives at blast date)
    """
    def __init__(self, earth, rf, vf, sc,
                 t0_lo, t0_hi, blast_mjd2k, mf_lo, mf_hi, nseg,
                 vinf_dep_ms):
        self.earth=earth; self.rf=rf; self.vf=vf; self.sc=sc
        self.t0_lo=t0_lo; self.t0_hi=t0_hi
        self.blast=blast_mjd2k
        self.mf_lo=mf_lo; self.mf_hi=mf_hi
        self.nseg=nseg
        self.vinf=vinf_dep_ms   # Falcon Heavy departure excess speed [m/s]

    def get_bounds(self):
        lb = [self.t0_lo] + [-1.0]*3*self.nseg + [self.mf_lo]
        ub = [self.t0_hi] + [ 1.0]*3*self.nseg + [self.mf_hi]
        return lb, ub

    def get_nec(self): return 7
    def get_nic(self): return self.nseg

    def fitness(self, x):
        t0  = x[0]
        thr = x[1:1+3*self.nseg]
        mf  = x[-1]
        tof = self.blast - t0

        if tof <= 0:
            return [1e9] + [1e9]*7 + [1e9]*self.nseg

        t0_ep = pk.epoch(t0,        'mjd2000')
        tf_ep = pk.epoch(self.blast,'mjd2000')

        # Earth state at departure
        r0, v0 = self.earth.eph(t0_ep)

        # Add Falcon Heavy C3 departure excess velocity in prograde direction.
        # Prograde = direction of Earth's velocity vector (tangential to orbit).
        # This is the most efficient departure direction for inner solar system.
        v0_arr   = np.array(v0)
        v0_hat   = v0_arr / np.linalg.norm(v0_arr)   # prograde unit vector
        v0_depart = v0_arr + self.vinf * v0_hat        # Earth v + C3 kick

        x0 = pk.sims_flanagan.sc_state(r0, tuple(v0_depart), self.sc.mass)
        xf = pk.sims_flanagan.sc_state(tuple(self.rf), tuple(self.vf), mf)

        leg = pk.sims_flanagan.leg(t0_ep, x0, thr, tf_ep, xf,
                                    self.sc, pk.MU_SUN)
        leg.high_fidelity = False

        # ── Objective: propellant + bang-bang regularisation ──────────
        # Primary: minimise propellant fraction
        prop_frac = (self.sc.mass - mf) / self.sc.mass

        # Regularisation: penalise partial throttle
        # um*(1-um) = 0 when um=0 or um=1, max at um=0.5
        # This pushes optimiser toward bang-bang (fully on or fully off)
        tmags = [np.linalg.norm(thr[3*k:3*k+3]) for k in range(self.nseg)]
        bang_bang_penalty = sum(um * (1.0 - um) for um in tmags) / self.nseg
        epsilon = 0.05   # weight: larger = more bang-bang, smaller = more fuel-optimal

        obj = prop_frac + epsilon * bang_bang_penalty

        mc = leg.mismatch_constraints()
        mc_n = [mc[0]/pk.AU,            mc[1]/pk.AU,            mc[2]/pk.AU,
                mc[3]/pk.EARTH_VELOCITY, mc[4]/pk.EARTH_VELOCITY,mc[5]/pk.EARTH_VELOCITY,
                mc[6]/self.sc.mass]

        tc = list(leg.throttles_constraints())
        return [obj] + mc_n + tc

    def gradient(self, x):
        return pg.estimate_gradient_h(lambda x_: self.fitness(x_), x)

# ── MAIN ──────────────────────────────────────────────────────────
print('='*60)
print(' Earth → 2024 PDC25  |  Low-Thrust  |  Rendezvous')
print(f' Blast date (fixed): {BLAST_DATE}')
print('='*60)

with open(SOL_FILE) as f:
    sol = json.load(f)

def to_mjd(iso):  return AstroTime(iso,scale='tdb',format='iso').tdb.mjd
def to_mjd2k(iso): return to_mjd(iso) - 51544.5

blast_mjd  = to_mjd(BLAST_DATE)
blast_mjd2k = to_mjd2k(BLAST_DATE)

# ONE grss call
print(f'\nPropagating asteroid to blast date...')
rf, vf = asteroid_state_at(sol, blast_mjd)
print(f'  |r| = {np.linalg.norm(rf)/AU_M:.5f} AU')
print(f'  |v| = {np.linalg.norm(vf)/1e3:.4f} km/s')
print(f'  grss done — zero grss calls during optimisation')

earth = pk.planet.jpl_lp('earth')
sc    = pk.sims_flanagan.spacecraft(M_WET, THRUST, ISP)
print(f'\nS/C: {M_WET:.0f} kg wet | {M_DRY:.0f} kg dry | {M_FUEL:.0f} kg xenon')
print(f'     {THRUST*1e3:.0f} mN (2× NEXT-C) | Isp {ISP} s | {NSEG} segments')
print(f'     Falcon Heavy C3 = {C3_KM2S2:.0f} km²/s²  →  v∞ = {VINF_MS/1e3:.3f} km/s (prograde kick at departure)')

t0_lo = to_mjd2k(T0_LOWER_ISO)
t0_hi = to_mjd2k(T0_UPPER_ISO)
tof_min = blast_mjd2k - t0_hi
tof_max = blast_mjd2k - t0_lo
print(f'\nLaunch window : {T0_LOWER_ISO} → {T0_UPPER_ISO}')
print(f'Blast date    : {BLAST_DATE} (fixed)')
print(f'TOF range     : {tof_min:.0f} – {tof_max:.0f} days '
      f'({tof_min/365.25:.1f} – {tof_max/365.25:.1f} yr)')

udp  = LowThrustUDP(earth, rf, vf, sc,
                     t0_lo, t0_hi, blast_mjd2k,
                     M_DRY, M_WET, NSEG,
                     vinf_dep_ms=VINF_MS)
prob = pg.problem(udp)
# Tolerances: mismatch normalised to O(1), so 1e-4 is ~15 km position error
# Throttle constraints already O(1)
prob.c_tol = [1e-4]*7 + [1e-4]*NSEG
print(f'\nDecision vector: {len(udp.get_bounds()[0])} vars')
print(f'Running {N_STARTS} random starts (pygmo NLopt SLSQP)...')

best_mf = -np.inf; champion_x = None

for seed in range(N_STARTS):
    pop = pg.population(prob, 1, seed=seed*13+RANDOM_SEED)
    uda = pg.nlopt('slsqp')
    uda.maxeval  = 12000   # more iterations
    uda.ftol_rel = 1e-9
    uda.xtol_rel = 1e-9
    try:
        pop = pg.algorithm(uda).evolve(pop)
    except Exception as e:
        print(f'  Seed {seed:2d}: error ({e})')
        continue

    feasible = prob.feasibility_f(pop.champion_f)
    prop_kg  = pop.champion_f[0] * sc.mass
    mf_try   = sc.mass - prop_kg
    # Show max mismatch violation to diagnose convergence
    fvec     = pop.champion_f
    mc_viol  = max(abs(fvec[1+i]) for i in range(7)) if len(fvec) > 7 else 999
    print(f'  Seed {seed:2d}: feasible={str(feasible):<5}  '
          f'prop={prop_kg:7.1f} kg  mf={mf_try:7.1f} kg  '
          f'mc_viol={mc_viol:.2e}')

    if feasible and mf_try > best_mf:
        best_mf = mf_try; champion_x = pop.champion_x.copy()

if champion_x is None:
    print('\nNo feasible solution. Try widening the launch window.')
    raise SystemExit(1)

t0_opt   = champion_x[0]
tof_opt  = blast_mjd2k - t0_opt
mf_opt   = champion_x[-1]
prop_opt = sc.mass - mf_opt
throttles = champion_x[1:1+3*NSEG]
tmags = [np.linalg.norm(throttles[3*k:3*k+3]) for k in range(NSEG)]
t0_iso = AstroTime(t0_opt+51544.5, format='mjd', scale='tdb').iso[:10]

print('\n' + '='*60)
print(' RESULT')
print('='*60)
print(f'  Launch date   : {t0_iso}')
print(f'  Blast date    : {BLAST_DATE} (fixed)')
print(f'  TOF           : {tof_opt:.1f} days  ({tof_opt/365.25:.2f} yr)')
print(f'  Wet mass      : {sc.mass:.0f} kg')
print(f'  Propellant    : {prop_opt:.1f} kg  ({100*prop_opt/sc.mass:.1f}%)')
print(f'  Unused fuel   : {M_FUEL-prop_opt:.1f} kg')
print(f'  Final mass    : {mf_opt:.1f} kg')
print(f'  Max throttle  : {max(tmags):.3f}')
print(f'  Mean throttle : {np.mean(tmags):.3f}')
print('='*60)


 Earth → 2024 PDC25  |  Low-Thrust  |  Rendezvous
 Blast date (fixed): 2033-11-01

Propagating asteroid to blast date...
  |r| = 2.28315 AU
  |v| = 15.4017 km/s
  grss done — zero grss calls during optimisation

S/C: 2401 kg wet | 1901 kg dry | 500 kg xenon
     472 mN (2× NEXT-C) | Isp 4190 s | 30 segments
     Falcon Heavy C3 = 15 km²/s²  →  v∞ = 3.873 km/s (prograde kick at departure)

Launch window : 2030-03-01 → 2031-10-01
Blast date    : 2033-11-01 (fixed)
TOF range     : 762 – 1341 days (2.1 – 3.7 yr)

Decision vector: 92 vars
Running 25 random starts (pygmo NLopt SLSQP)...
  Seed  0: feasible=False  prop=  498.2 kg  mf= 1902.9 kg  mc_viol=6.95e-02
  Seed  1: feasible=False  prop=  507.0 kg  mf= 1894.1 kg  mc_viol=2.44e-02
  Seed  2: feasible=False  prop=  343.2 kg  mf= 2057.9 kg  mc_viol=1.93e-01
  Seed  3: feasible=False  prop=  126.0 kg  mf= 2275.1 kg  mc_viol=2.02e+00
  Seed  4: feasible=False  prop=  359.7 kg  mf= 2041.4 kg  mc_viol=5.65e-02
  Seed  5: feasible=False  prop=